# Experimentos 1 y 3: ¿cambia el entrelazamiento la gravedad?

**Proyecto:** *Vacuum Particle Journey*, línea A (mecanismo físico).

La revisión propone convertir el salto de la hipótesis en el objeto de estudio:

> ¿Puede la estructura del entrelazamiento modificar la geometría gravitatoria de un sistema de una manera que **no** quede ya explicada por su distribución ordinaria de energía y momento?

Este notebook no usa curvas de rotación ni parámetros astrofísicos. Trabaja con los sistemas cuánticos más sencillos en los que la pregunta tiene respuesta exacta: cadenas de osciladores (estados gaussianos) y pocos qubits.

**Tres hipótesis gravitatorias explícitas que se ponen a prueba:**

| | Hipótesis | La geometría depende de |
|---|---|---|
| **H0** | Gravedad semiclásica: $G_{\mu\nu} = 8\pi G\,\langle T_{\mu\nu}\rangle$ | la energía local |
| **H1** | Holografía (Ryu–Takayanagi, ER = EPR) | áreas y conectividad fijadas por el entrelazamiento |
| **H2** | "El entrelazamiento pesa": $\rho_{grav} = \langle T_{00}\rangle + \varepsilon\, I(A{:}B)/V$ | la energía local **más** un término de información mutua |

**Secciones:**
1. Dos estados con **la misma energía local** y distinto entrelazamiento (estado termofield doble frente a dos estados térmicos independientes).
2. **Geometría emergente** a partir de la información mutua, en 3D: ¿aparece un "puente" entre las dos cadenas?
3. **Dónde vive la energía del entrelazamiento:** romperlo cuesta energía localizada en el corte, que ya está en $\langle T_{\mu\nu}\rangle$ (control de doble conteo).
4. **Prueba de conservación de H2:** ¿puede un término proporcional al entrelazamiento conservar la energía?
5. **Experimento 3, decoherencia:** ¿destruye el entorno el entrelazamiento o lo redistribuye?
6. **El puente matemático:** la "primera ley del entrelazamiento" ($\delta S = \delta\langle K\rangle$), el paso que en holografía produce las ecuaciones de Einstein linealizadas (Faulkner et al. 2014; Jacobson 2016).

> Todo corre en segundos. Solo usa numpy, scipy, matplotlib y plotly (incluido en Colab).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.linalg import logm
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2,
                     "font.size": 10, "legend.frameon": False})
C_L, C_R, C_ACC, C_GREY, C_WARN = "#eb6834", "#2a78d6", "#1baf7a", "#8a8a85", "#e34948"

## 0. Herramientas: estados gaussianos de una cadena de osciladores

Cadena de $N$ osciladores acoplados, $H = \tfrac12\sum_i\big[p_i^2 + \omega_0^2 x_i^2\big] + \tfrac{k}{2}\sum_i (x_i - x_{i+1})^2$, con $\hbar = 1$. Un estado gaussiano queda descrito por sus covarianzas $X_{ij} = \langle x_i x_j\rangle$ y $P_{ij} = \langle p_i p_j\rangle$. La entropía de una región $A$ sale de los valores propios simplécticos $\nu$ de $\sqrt{X_A P_A}$:
$$S_A = \sum_\nu \Big[(\nu+\tfrac12)\ln(\nu+\tfrac12) - (\nu-\tfrac12)\ln(\nu-\tfrac12)\Big]$$

**Estado termofield doble (TFD)** de dos cadenas idénticas L y R: es un estado *puro* cuyo estado reducido en cada cadena es exactamente el térmico. Modo a modo es un estado comprimido de dos modos con $\cosh 2r = \coth(\beta\Omega/2)$.

In [ ]:
def chain_K(N, w0=0.3, k=1.0):
    K = np.diag(np.full(N, w0**2 + 2 * k)) - k * np.eye(N, k=1) - k * np.eye(N, k=-1)
    K[0, 0] -= k; K[-1, -1] -= k                     # extremos libres
    return K

def modes(K):
    Om2, U = np.linalg.eigh(K)
    return np.sqrt(Om2), U

def cov_thermal(K, beta=np.inf):
    Om, U = modes(K)
    c = np.ones_like(Om) if np.isinf(beta) else 1 / np.tanh(beta * Om / 2)
    return U @ np.diag(c / (2 * Om)) @ U.T, U @ np.diag(c * Om / 2) @ U.T

def cov_tfd(K, beta):
    Om, U = modes(K)
    ch, sh = 1 / np.tanh(beta * Om / 2), 1 / np.sinh(beta * Om / 2)
    XLL, XLR = U @ np.diag(ch / (2 * Om)) @ U.T, U @ np.diag(sh / (2 * Om)) @ U.T
    PLL, PLR = U @ np.diag(ch * Om / 2) @ U.T, -U @ np.diag(sh * Om / 2) @ U.T
    return np.block([[XLL, XLR], [XLR, XLL]]), np.block([[PLL, PLR], [PLR, PLL]])

def cov_product(XA, PA, XB, PB):
    Z = np.zeros((len(XA), len(XB)))
    return np.block([[XA, Z], [Z.T, XB]]), np.block([[PA, Z], [Z.T, PB]])

def entropy(X, P, idx):
    idx = np.asarray(idx)
    nu = np.sqrt(np.clip(np.linalg.eigvals(X[np.ix_(idx, idx)] @ P[np.ix_(idx, idx)]).real, 0.25, None))
    a, b = nu + 0.5, np.clip(nu - 0.5, 1e-300, None)
    return float(np.sum(a * np.log(a) - np.where(nu - 0.5 > 1e-12, b * np.log(b), 0.0)))

def mutual_info(X, P, A, B):
    return entropy(X, P, A) + entropy(X, P, B) - entropy(X, P, list(A) + list(B))

def energy_profile(X, P, idx, w0, k):
    # energía por sitio de una cadena (cada enlace se reparte a medias entre sus dos sitios)
    idx = list(idx); e = 0.5 * np.array([P[i, i] + w0**2 * X[i, i] for i in idx])
    for a in range(len(idx) - 1):
        i, j = idx[a], idx[a + 1]
        bond = 0.5 * k * (X[i, i] + X[j, j] - 2 * X[i, j])
        e[a] += bond / 2; e[a + 1] += bond / 2
    return e

N, W0, KK, BETA = 24, 0.3, 1.0, 2.0
K = chain_K(N, W0, KK)
L, R = list(range(N)), list(range(N, 2 * N))
Xt, Pt = cov_thermal(K, BETA)
X_tfd, P_tfd = cov_tfd(K, BETA)
X_prod, P_prod = cov_product(Xt, Pt, Xt, Pt)
print(f"Comprobación: el TFD es puro → S(L∪R) = {entropy(X_tfd, P_tfd, L + R):.2e} (debe ser 0)")

## 1. Misma energía local, distinto entrelazamiento

Comparamos el **TFD** (L y R entrelazadas) con **térmico ⊗ térmico** (L y R independientes). En cada cadena, los dos estados son localmente idénticos: misma densidad de energía sitio a sitio, mismas correlaciones internas. Lo único que cambia es la correlación *entre* L y R.

**Consecuencia para H0:** si la gravedad solo depende de $\langle T_{\mu\nu}\rangle$ local, los dos estados producen exactamente la misma geometría. Cualquier diferencia gravitatoria tiene que venir de algo **no local**.

In [ ]:
eL_tfd, eL_prod = energy_profile(X_tfd, P_tfd, L, W0, KK), energy_profile(X_prod, P_prod, L, W0, KK)
eR_tfd, eR_prod = energy_profile(X_tfd, P_tfd, R, W0, KK), energy_profile(X_prod, P_prod, R, W0, KK)
I_tfd, I_prod = mutual_info(X_tfd, P_tfd, L, R), mutual_info(X_prod, P_prod, L, R)
S_th = entropy(Xt, Pt, list(range(N)))
print(f"Máxima diferencia de energía local entre los dos estados: {max(np.abs(eL_tfd-eL_prod).max(), np.abs(eR_tfd-eR_prod).max()):.1e}")
print(f"Información mutua I(L:R): TFD = {I_tfd:.3f} nats (= 2·S_térmica = {2*S_th:.3f}),  térmico⊗térmico = {I_prod:.1e}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(eL_prod, "o", ms=7, color=C_GREY, label="térmico ⊗ térmico")
a1.plot(eL_tfd, "x", ms=6, color=C_ACC, label="TFD (entrelazado)")
a1.set_xlabel("sitio de la cadena L"); a1.set_ylabel("energía local"); a1.set_title("La energía local es idéntica", loc="left"); a1.legend()
betas = np.geomspace(0.3, 20, 30)
a2.semilogx(1 / betas, [mutual_info(*cov_tfd(K, b), L, R) for b in betas], color=C_ACC, label="TFD")
a2.semilogx(1 / betas, np.zeros_like(betas), color=C_GREY, label="térmico ⊗ térmico")
a2.set_xlabel("temperatura T = 1/β"); a2.set_ylabel("I(L:R) [nats]"); a2.set_title("Pero el entrelazamiento L–R no", loc="left"); a2.legend()
plt.tight_layout(); plt.show()

## 2. Geometría emergente a partir del entrelazamiento (vista 3D)

Siguiendo la idea de *Space from Hilbert space* (Cao, Carroll y Michalakis, 2017), definimos una distancia entre dos sitios a partir de su información mutua: mucho entrelazamiento = cerca.
$$d(a,b) = -\ln\frac{I(a{:}b)}{I_{max}}$$
y colocamos los $2N$ sitios en 3D con escalado multidimensional (MDS), que busca posiciones cuyas distancias se parezcan a $d$.

- **Validación:** una sola cadena en su estado fundamental debe salir como una línea.
- **Térmico ⊗ térmico:** dos cadenas separadas, sin conexión.
- **TFD a baja temperatura:** las dos cadenas aparecen unidas sitio a sitio por "puentes" (en verde): la imagen de ER = EPR en este modelo.
- **TFD a más temperatura:** el entrelazamiento entre cadenas supera al de cada sitio con su vecino, y las dos cadenas se **funden** en un único espacio: cada sitio queda más cerca de su pareja que de su vecino.

Esto ilustra **H1**: el entrelazamiento cambia la *conectividad* de la geometría, no la energía.

In [ ]:
def pair_mi(X, P, sites):
    n = len(sites); M = np.zeros((n, n)); Ss = [entropy(X, P, [s]) for s in sites]
    for a in range(n):
        for b in range(a + 1, n):
            M[a, b] = M[b, a] = Ss[a] + Ss[b] - entropy(X, P, [sites[a], sites[b]])
    return M

def mds3(M, cap=12.0):
    Imax = M.max(); Dm = np.where(M > 1e-12, -np.log(np.clip(M, 1e-300, None) / Imax), cap)
    Dm = np.minimum(Dm, cap); np.fill_diagonal(Dm, 0)
    n = len(Dm); J = np.eye(n) - 1 / n; Bm = -0.5 * J @ (Dm**2) @ J
    w, V = np.linalg.eigh(Bm); o = np.argsort(w)[::-1][:3]
    return V[:, o] * np.sqrt(np.clip(w[o], 0, None)), Dm

Nv = 16; Kv = chain_K(Nv, W0, KK); Lv, Rv = list(range(Nv)), list(range(Nv, 2 * Nv))
Xg, Pg = cov_thermal(Kv)
Xtv, Ptv = cov_thermal(Kv, BETA)
CASES = {"Una cadena, estado fundamental (validación)": (Xg, Pg, Lv),
         "Térmico ⊗ térmico": (*cov_product(Xtv, Ptv, Xtv, Ptv), Lv + Rv),
         "TFD a baja temperatura (T = 0.125): el puente se ve": (*cov_tfd(Kv, 8.0), Lv + Rv),
         "TFD a T = 0.5: las dos cadenas se funden": (*cov_tfd(Kv, BETA), Lv + Rv)}
EMB = {}
for name, (X_, P_, sites) in CASES.items():
    M = pair_mi(X_, P_, sites); Y, Dm = mds3(M); EMB[name] = (Y, M, len(sites))
    if len(sites) == 2 * Nv:
        dLR = np.mean([Dm[i, Nv + i] for i in range(Nv)]); dLL = np.mean([Dm[i, i + 1] for i in range(Nv - 1)])
        print(f"{name}: distancia media L_i–R_i = {dLR:.2f}, entre vecinos de una cadena = {dLL:.2f}")

def show3d(name):
    Y, M, n = EMB[name]; two = n > Nv
    thr = np.sort(M[np.triu_indices(n, 1)])[-int(2.2 * n)]
    bridge = [(a, Nv + a) for a in range(Nv)] if two else []
    segs = [(a, b) for a in range(n) for b in range(a + 1, n) if M[a, b] >= thr and M[a, b] > 1e-9 and (a, b) not in bridge]
    def lines(pairs):
        xs, ys, zs = [], [], []
        for a, b in pairs:
            xs += [Y[a, 0], Y[b, 0], None]; ys += [Y[a, 1], Y[b, 1], None]; zs += [Y[a, 2], Y[b, 2], None]
        return xs, ys, zs
    if HAVE_PLOTLY:
        tr = []
        xs, ys, zs = lines(segs)
        tr.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="#c3c2b7", width=3), name="entrelazamiento dentro de una cadena"))
        if two and np.mean([M[a, b] for a, b in bridge]) > 1e-9:
            xs, ys, zs = lines(bridge)
            tr.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=C_ACC, width=5), name="puente L–R (entrelazamiento entre cadenas)"))
        tr.append(go.Scatter3d(x=Y[:Nv, 0], y=Y[:Nv, 1], z=Y[:Nv, 2], mode="markers", marker=dict(size=6, color=C_L, symbol="circle"),
                               text=[f"L{i}" for i in range(Nv)], name="cadena L"))
        if two:
            tr.append(go.Scatter3d(x=Y[Nv:, 0], y=Y[Nv:, 1], z=Y[Nv:, 2], mode="markers", marker=dict(size=5, color=C_R, symbol="diamond"),
                                   text=[f"R{i}" for i in range(Nv)], name="cadena R"))
        fig = go.Figure(tr)
        fig.update_layout(title=f"Geometría emergente: {name}", height=480, margin=dict(l=0, r=0, t=40, b=0),
                          scene=dict(aspectmode="data", xaxis_title="", yaxis_title="", zaxis_title=""))
        fig.show()
    else:
        f = plt.figure(figsize=(5.5, 4.5)); ax = f.add_subplot(projection="3d")
        for a, b in segs: ax.plot(*Y[[a, b]].T, color="#c3c2b7", lw=0.8)
        for a, b in bridge: ax.plot(*Y[[a, b]].T, color=C_ACC, lw=1.5)
        ax.scatter(*Y[:Nv].T, c=C_L, s=25, marker="o")
        if two: ax.scatter(*Y[Nv:].T, c=C_R, s=25, marker="D")
        ax.set_title(name, loc="left"); plt.show()

for name in CASES:
    show3d(name)

In [ ]:
# ¿Cómo cambia el "puente" con la temperatura? En holografía, un agujero negro más caliente/grande = más entrelazamiento.
rows = []
for b in [8, 4, 2, 1, 0.5]:
    M = pair_mi(*cov_tfd(Kv, b), Lv + Rv); _, Dm = mds3(M)
    rows.append(dict(T=round(1 / b, 3), S_térmica=round(entropy(*cov_thermal(Kv, b), Lv), 3),
                     I_Li_Ri=round(np.mean([M[i, Nv + i] for i in range(Nv)]), 4),
                     distancia_puente=round(np.mean([Dm[i, Nv + i] for i in range(Nv)]), 2),
                     distancia_vecinos=round(np.mean([Dm[i, i + 1] for i in range(Nv - 1)]), 2)))
print(pd.DataFrame(rows).to_string(index=False))

## 3. ¿Dónde vive la energía del entrelazamiento? (control de doble conteo)

Tomamos una cadena de $2N$ sitios en su estado fundamental, con mucho entrelazamiento entre su mitad izquierda y su mitad derecha, y lo **cortamos** de dos maneras:
- **Corte mixto:** cada mitad conserva su estado reducido, pero se borran las correlaciones entre ellas.
- **Corte puro:** cada mitad pasa al estado fundamental de su propio Hamiltoniano, desacoplada.

Si cortar el entrelazamiento cuesta energía, y esa energía aparece **localizada en el corte**, entonces la energía asociada al entrelazamiento ya forma parte de $\langle T_{\mu\nu}\rangle$. Un término gravitatorio extra proporcional al entrelazamiento la contaría dos veces. Es la advertencia de la revisión sobre no volver a contar la energía de la interacción fuerte.

In [ ]:
N2 = 2 * N; K2 = chain_K(N2, W0, KK); A_, B_ = list(range(N)), list(range(N, N2))
Xg2, Pg2 = cov_thermal(K2)
Xmix, Pmix = Xg2.copy(), Pg2.copy()
Xmix[np.ix_(A_, B_)] = 0; Xmix[np.ix_(B_, A_)] = 0; Pmix[np.ix_(A_, B_)] = 0; Pmix[np.ix_(B_, A_)] = 0
Kcut = K2.copy(); Kcut[N - 1, N] = Kcut[N, N - 1] = 0; Kcut[N - 1, N - 1] -= KK; Kcut[N, N] -= KK
Xpure, Ppure = cov_thermal(Kcut)
e0 = energy_profile(Xg2, Pg2, range(N2), W0, KK)
d_mix, d_pure = energy_profile(Xmix, Pmix, range(N2), W0, KK) - e0, energy_profile(Xpure, Ppure, range(N2), W0, KK) - e0
S_half = entropy(Xg2, Pg2, A_)
print(f"Entrelazamiento entre mitades S(A) = {S_half:.3f} nats")
print(f"Coste de cortarlo: mixto ΔE = {d_mix.sum():.4f}, puro ΔE = {d_pure.sum():.4f} (en unidades de ħω)")
print(f"Fracción de la energía del corte puro dentro de ±3 sitios del corte: {np.abs(d_pure[N-3:N+3]).sum()/np.abs(d_pure).sum():.0%}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.arange(N2) - (N - 0.5)
a1.bar(x, d_mix, width=0.8, color=C_GREY, label="corte mixto")
a1.plot(x, d_pure, "o-", ms=3, color=C_ACC, label="corte puro")
a1.axvline(0, color="#c3c2b7", ls="--", lw=1); a1.set_xlabel("sitio respecto al corte"); a1.set_ylabel("ΔE por sitio")
a1.set_title("Romper el entrelazamiento cuesta energía en el corte", loc="left"); a1.legend()
w0s = np.geomspace(0.05, 2, 20); Ss, Es = [], []
for w in w0s:
    Kw = chain_K(N2, w, KK); Xw, Pw = cov_thermal(Kw); Ss.append(entropy(Xw, Pw, A_))
    Kc = Kw.copy(); Kc[N - 1, N] = Kc[N, N - 1] = 0; Kc[N - 1, N - 1] -= KK; Kc[N, N] -= KK
    Xc, Pc = cov_thermal(Kc); Es.append((energy_profile(Xc, Pc, range(N2), w, KK) - energy_profile(Xw, Pw, range(N2), w, KK)).sum())
a2.plot(Ss, Es, "o-", ms=4, color=C_ACC); a2.set_xlabel("entrelazamiento S(A) [nats]"); a2.set_ylabel("energía del corte puro")
a2.set_title("Más entrelazamiento, más energía en la frontera", loc="left")
plt.tight_layout(); plt.show()

## 4. Prueba de conservación de H2 ("el entrelazamiento pesa")

Dos qubits con $H = \tfrac{\omega}{2}(\sigma^z_L + \sigma^z_R)$ en estado TFD. Hacemos una **medida de la energía de L**: la energía media no cambia (la medida conmuta con $H$), pero el entrelazamiento desaparece y solo queda correlación clásica.

Bajo H2, la "masa gravitatoria" cambiaría sin que ninguna energía entre ni salga. Eso viola la conservación ($\nabla_\mu T^{\mu\nu} = 0$, que la identidad de Bianchi exige en cualquier teoría tipo Einstein). Por tanto, **un término gravitatorio proporcional al entrelazamiento no puede ser un término de energía local añadido**: o ya está en $\langle T_{\mu\nu}\rangle$ (sección 3), o actúa sobre la geometría de otra forma (no local, como en H1).

In [ ]:
def ent(rho):
    w = np.linalg.eigvalsh(rho); w = w[w > 1e-14]; return float(-np.sum(w * np.log(w)))
def ptrace2(rho, keep):   # dos qubits
    r = rho.reshape(2, 2, 2, 2)
    return np.trace(r, axis1=1, axis2=3) if keep == 0 else np.trace(r, axis1=0, axis2=2)
sz = np.diag([1.0, -1.0]); I2 = np.eye(2); om = 1.0
H = 0.5 * om * (np.kron(sz, I2) + np.kron(I2, sz))
rows = []
for beta in [0.5, 1, 2, 4]:
    E = np.array([0.5 * om, -0.5 * om]); p = np.exp(-beta * E); p /= p.sum()
    psi = np.zeros(4); psi[0], psi[3] = np.sqrt(p[0]), np.sqrt(p[1])            # Σ sqrt(p_n)|n>|n>
    rho = np.outer(psi, psi)
    P0, P1 = np.kron(np.diag([1, 0]), I2), np.kron(np.diag([0, 1]), I2)
    rho_m = P0 @ rho @ P0 + P1 @ rho @ P1                                       # medida de la energía de L
    I_b = ent(ptrace2(rho, 0)) + ent(ptrace2(rho, 1)) - ent(rho)
    I_a = ent(ptrace2(rho_m, 0)) + ent(ptrace2(rho_m, 1)) - ent(rho_m)
    rows.append(dict(β=beta, E_antes=round(np.trace(H @ rho).real, 6), E_después=round(np.trace(H @ rho_m).real, 6),
                     I_antes=round(I_b, 4), I_después=round(I_a, 4), cambio_masa_H2_en_ε=round(I_a - I_b, 4)))
print(pd.DataFrame(rows).to_string(index=False))
print("\n→ La energía se conserva exactamente; la información mutua cae a la mitad (queda solo la parte clásica).")
print("  Bajo H2 la masa cambiaría en ε·ΔI sin flujo de energía: H2 no conserva la energía como término local.")

## 5. Experimento 3: ¿destruye la decoherencia el entrelazamiento o lo redistribuye?

Un par de Bell entre A y B. A interacciona, una a una, con $n$ partículas de un entorno E (un modelo de colisiones). La intensidad de cada colisión, $\theta$, representa lo denso o caliente que es el medio. Medimos:
- la **negatividad** de AB (entrelazamiento genuino entre A y B);
- la información mutua $I(A{:}B)$ y la información mutua entre A y el entorno, $I(A{:}E)$.

Para un estado global puro se cumple exactamente $I(A{:}B) + I(A{:}E) = 2\,S(A)$. Si $S(A)$ no cambia, **la correlación no se destruye: se traslada al entorno**. Es la advertencia de la revisión: no se puede suponer que más temperatura implique menos gravedad de origen cuántico. Depende de *qué* entrelazamiento se supone que gravita.

In [ ]:
def partial_trace(psi, keep, n):
    psi = psi.reshape([2] * n); rest = [i for i in range(n) if i not in keep]
    m = np.transpose(psi, keep + rest).reshape(2 ** len(keep), -1)
    return m @ m.conj().T
def negativity(rho_ab):
    r = rho_ab.reshape(2, 2, 2, 2).transpose(0, 3, 2, 1).reshape(4, 4)    # transpuesta parcial sobre B
    return float(np.sum(np.abs(np.linalg.eigvalsh(r))) - 1) / 2
def apply_2q(psi, U, a, b, n):
    psi = psi.reshape([2] * n); order = [a, b] + [i for i in range(n) if i not in (a, b)]
    m = np.transpose(psi, order).reshape(4, -1); m = U @ m
    return np.transpose(m.reshape([2] * n), np.argsort(order)).reshape(-1)

def collision_run(theta, n_env=8):
    n = 2 + n_env
    psi = np.zeros(2 ** n, complex); bell = np.zeros(4); bell[0] = bell[3] = 1 / np.sqrt(2)
    psi = np.kron(bell, np.eye(2 ** n_env)[0]).astype(complex)
    # colisión: rotación controlada (A controla una partícula del entorno) → decoherencia de fase de A
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    U = np.eye(4, dtype=complex); U[2:, 2:] = [[c, -1j * s], [-1j * s, c]]
    out = []
    for step in range(n_env + 1):
        rAB = partial_trace(psi, [0, 1], n); rA = partial_trace(psi, [0], n); rB = partial_trace(psi, [1], n)
        env = list(range(2, n)); rE = partial_trace(psi, env, n); rAE = partial_trace(psi, [0] + env, n)
        SA, SB, SAB, SE, SAE = ent(rA), ent(rB), ent(rAB), ent(rE), ent(rAE)
        out.append(dict(colisiones=step, negatividad_AB=negativity(rAB), I_AB=SA + SB - SAB, I_AE=SA + SE - SAE, dosSA=2 * SA))
        if step < n_env:
            psi = apply_2q(psi, U, 0, 2 + step, n)
    return pd.DataFrame(out)

fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for theta, col, lab in [(0.3, "#a8e6cf", "θ = 0.3 (medio frío, diluido)"), (0.8, C_ACC, "θ = 0.8"), (1.6, "#0b6b4a", "θ = 1.6 (caliente, denso)")]:
    d = collision_run(theta)
    axs[0].plot(d.colisiones, d.negatividad_AB, "o-", ms=3, color=col, label=lab)
    axs[1].plot(d.colisiones, d.I_AB, "o-", ms=3, color=col)
    axs[2].plot(d.colisiones, d.I_AE, "o-", ms=3, color=col)
    print(f"θ = {theta}: máx |I(A:B) + I(A:E) − 2S(A)| = {np.max(np.abs(d.I_AB + d.I_AE - d.dosSA)):.1e}")
axs[0].set_title("Entrelazamiento A–B (negatividad)", loc="left"); axs[1].set_title("Información mutua I(A:B)", loc="left")
axs[2].set_title("Información mutua con el entorno I(A:E)", loc="left")
for a in axs: a.set_xlabel("colisiones con el entorno")
axs[0].legend(fontsize=8.5); plt.tight_layout(); plt.show()

## 6. El puente matemático: la primera ley del entrelazamiento

Para cualquier región $A$ y cualquier perturbación pequeña de un estado $\rho_0$:
$$\delta S_A = \delta\langle K_A\rangle,\qquad K_A = -\ln\rho_{0,A}\ \ \text{(Hamiltoniano modular)}$$
La diferencia es la entropía relativa, de segundo orden en la perturbación. En teorías holográficas, aplicar esta identidad a todas las regiones esféricas equivale a las **ecuaciones de Einstein linealizadas** (Faulkner et al. 2014). Jacobson (2016) llega a la ecuación completa con un principio de "equilibrio de entrelazamiento".

Lo comprobamos en una cadena de Ising con campo transverso (10 espines, región A = 4 espines). Es la relación que una ecuación gravitatoria salida del entrelazamiento tendría que generalizar. **Ojo:** por sí sola, reproduce la gravedad de Einstein, sin ningún término extra.

In [ ]:
def tfim_ground(n, h):
    sx, szz = np.array([[0, 1], [1, 0]]), np.diag([1.0, -1.0])
    def op(o, i):
        m = np.array([[1.0]])
        for j in range(n): m = np.kron(m, o if j == i else np.eye(2))
        return m
    Hm = -sum(op(szz, i) @ op(szz, i + 1) for i in range(n - 1)) - h * sum(op(sx, i) for i in range(n))
    w, V = np.linalg.eigh(Hm); return V[:, 0]
def rho_A(psi, nA, n):
    m = psi.reshape(2 ** nA, 2 ** (n - nA)); return m @ m.T
n_s, nA, h0 = 10, 4, 1.0
rho0 = rho_A(tfim_ground(n_s, h0), nA, n_s)
w, V = np.linalg.eigh(rho0); Kmod = V @ np.diag(-np.log(np.clip(w, 1e-14, None))) @ V.T
rows = []
for dh in [0.08, 0.04, 0.02, 0.01, 0.005, 0.0025]:
    r1 = rho_A(tfim_ground(n_s, h0 + dh), nA, n_s)
    dS, dK = ent(r1) - ent(rho0), np.trace((r1 - rho0) @ Kmod)
    rows.append(dict(δh=dh, δS=dS, δK=dK, cociente=dS / dK, diferencia=abs(dK - dS)))
FL = pd.DataFrame(rows)
slope = np.polyfit(np.log(FL.δh), np.log(FL.diferencia), 1)[0]
print(FL.to_string(index=False))
print(f"\nδS/δ⟨K⟩ → 1 al reducir la perturbación; la diferencia escala como δh^{slope:.2f} (la entropía relativa es de orden 2) ✓")

## 7. Qué dicen estos modelos sobre la hipótesis

Resumen de lo que se ha calculado (en modelos mínimos; no son una prueba sobre el universo real):

| Pregunta | Resultado en estos modelos | Consecuencia |
|---|---|---|
| ¿Distinto entrelazamiento con la misma energía local cambia la gravedad? | Con H0, no: TFD y térmico⊗térmico tienen la misma $\langle T\rangle$ | El efecto, si existe, tiene que ser **no local** |
| ¿Puede el entrelazamiento crear geometría? | Sí, como **conectividad** (sección 2): aparece un puente L–R | Apoya ER = EPR como cambio de geometría, no como masa extra |
| ¿Dónde está la energía del entrelazamiento? | Localizada en las fronteras, dentro de $\langle T\rangle$ | Un término extra la contaría **dos veces** |
| ¿Puede "pesar" el entrelazamiento como término local (H2)? | No conserva la energía | H2 descartada en su forma simple |
| ¿La decoherencia destruye el entrelazamiento? | Lo redistribuye: $I(A{:}B)+I(A{:}E) = 2S(A)$ | La regla "caliente = menos puentes" exige elegir qué entrelazamiento gravita |
| ¿Cómo se pasa de entrelazamiento a gravedad? | Primera ley $\delta S = \delta\langle K\rangle$ verificada | Por sí sola da Einstein, sin extra |

**Lectura honesta:** el camino que sigue abierto para la hipótesis no es "el entrelazamiento añade masa", sino **"el entrelazamiento modifica la geometría de forma no local"** (como en H1), con una corrección a la primera ley que no esté ya contenida en $\langle T_{\mu\nu}\rangle$ y que conserve la energía-momento. Es el resultado teórico que la revisión pide: si una corrección así existe y se puede calcular su magnitud, la escala λ ≈ 0.7 kpc de los notebooks de galaxias tendría un candidato a origen.

**Siguientes pasos posibles:**
1. Calcular la corrección de segundo orden a la primera ley (la entropía relativa) en un modelo holográfico sencillo, y ver si se comporta como un término no local con una escala propia.
2. Repetir la geometría emergente con más dimensiones y comprobar si la distancia emergente obedece algún análogo de la ecuación de Poisson.
3. Definir con precisión qué medida de entrelazamiento (negatividad, entropía de formación, información mutua) entraría en la geometría, a la luz de la sección 5.

**Referencias:** Van Raamsdonk (2010) arXiv:1005.3035 · Ryu y Takayanagi (2006) arXiv:hep-th/0603001 · Maldacena y Susskind (2013) arXiv:1306.0533 · Faulkner, Guica, Hartman, Myers y Van Raamsdonk (2014) arXiv:1312.7856 · Jacobson (2016) arXiv:1505.04753 · Cao, Carroll y Michalakis (2017) arXiv:1606.08444 · Bose et al. (2017) arXiv:1707.06050 · Marletto y Vedral (2017) arXiv:1707.06036 · Hensen et al. (2015), *Nature* 526, 682.